# 5-1절 연습 문제 풀이

이 노트북은 5-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch05/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
DATA_ROOT = '../../downloads'

def loaders(dataset='MNIST', batch_size=64, transform=None, train_transform=None):
    cls = getattr(datasets, dataset)
    transform = transform or transforms.ToTensor()
    full = cls(root=DATA_ROOT, train=True, download=True,
               transform=train_transform or transform)
    test_set = cls(root=DATA_ROOT, train=False, download=True, transform=transform)
    n_valid = int(len(full) * 0.2)
    g = torch.Generator().manual_seed(SEED)
    tr, va = random_split(full, [len(full) - n_valid, n_valid], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size),
            DataLoader(test_set, batch_size=batch_size))

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, correct / n * 100

def fit(model, epochs=5, lr=1e-3, dataset='MNIST', **kw):
    tr, va, te = loaders(dataset, **kw)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        trl, tra = run_epoch(model, tr, criterion, optimizer)
        val, vaa = run_epoch(model, va, criterion)
        print(f'  {e}/{epochs} 훈련 {trl:.4f} / 검증 {val:.4f} ({vaa:.2f}%)')
    tel, tea = run_epoch(model, te, criterion)
    print(f'  평가 정확도 {tea:.2f}%')
    return tea

## 연습 5-1

다음 세 필터로 탐지할 수 있는 특징은 어떤 형태일까?

필터 1: -101-101-101, 필터 2: -11-1-11-1-11-1, 필터 3: 1-111-111-11

필터가 탐지할 수 있는 특징의 형태를 예상한 후, [코드 5-2]를 참고해 숫자마다 세 필터를 사용해 만든 특징 지도를 출력해 보자. 그리고 본문의 세로 방향 경계 필터로 만든 특징 지도와 어떤 차이가 있는지 비교해 보자. 특징 지도의 시각화 방법은 깃허브 예제 노트북을 참고하면 된다.

In [ ]:
# 세 필터를 3x3으로 만들어 특징 지도를 확인한다.
filters = {
    '필터 1': [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],   # 세로 경계(좌->우 밝아짐)
    '필터 2': [[-1, 1, -1], [-1, 1, -1], [-1, 1, -1]], # 세로 선(가는 줄기)
    '필터 3': [[1, -1, 1], [1, -1, 1], [1, -1, 1]],    # 세로 선의 반전
}
train_set = datasets.MNIST(root=DATA_ROOT, train=True, download=True,
                           transform=transforms.ToTensor())
image = train_set[0][0].unsqueeze(0).to(device)      # (1, 1, 28, 28)

maps, titles = [image.squeeze().cpu()], ['원본']
for name, w in filters.items():
    kernel = torch.tensor(w, dtype=torch.float32).view(1, 1, 3, 3).to(device)
    fmap = nn.functional.conv2d(image, kernel, padding=1)
    maps.append(fmap.squeeze().detach().cpu()); titles.append(name)
viz.plot_images(maps, titles, images_per_row=4)

- **필터 1**은 좌우 명암 차이를 재는 **세로 방향 경계** 필터로, 본문의 세로 경계 필터와 성격이 같다.
- **필터 2**는 가운데 열만 +1이라 **가는 세로 선**에 크게 반응한다. 경계(밝기가 한 번 바뀌는 지점)가 아니라 **선**(밝은 줄기)을 찾는다.
- **필터 3**은 필터 2의 부호를 뒤집은 형태라 **어두운 세로 선**에 반응한다. 두 특징 지도는 밝고 어두운 영역이 서로 반대로 나타난다.

## 연습 5-2

가로 방향 경계선과 대각선 방향 경계선을 탐지하는 필터를 직접 설계하고, [코드 5-2]를 참고해 MNIST 데이터셋 샘플의 특징 지도를 출력해 보자.

In [ ]:
designed = {
    '가로 경계': [[-1, -1, -1], [0, 0, 0], [1, 1, 1]],      # 위아래 명암 차
    '대각선(\\)': [[2, -1, -1], [-1, 2, -1], [-1, -1, 2]],   # 좌상 -> 우하
    '대각선(/)': [[-1, -1, 2], [-1, 2, -1], [2, -1, -1]],    # 우상 -> 좌하
}
maps, titles = [image.squeeze().cpu()], ['원본']
for name, w in designed.items():
    kernel = torch.tensor(w, dtype=torch.float32).view(1, 1, 3, 3).to(device)
    maps.append(nn.functional.conv2d(image, kernel, padding=1).squeeze().detach().cpu())
    titles.append(name)
viz.plot_images(maps, titles, images_per_row=4)

경계 필터의 원리는 **한 방향으로는 부호가 바뀌고, 그 직각 방향으로는 값이 같은** 형태다. 세로 경계는 좌우로 부호가 바뀌고, 가로 경계는 위아래로 바뀐다. 대각선 필터는 대각 성분만 양수로 두어 그 방향의 선에 크게 반응하게 만든다.

## 연습 5-3

다음에 제시된 조건이 변화했을 때 합성곱 신경망의 파라미터 수가 어떻게 바뀌는지 설명해 보자.

특징 탐지, 요약, 조합 과정의 반복 횟수를 늘렸을 때

합성곱 필터의 크기를 늘렸을 때

풀링의 커널 크기를 늘렸을 때

합성곱 필터의 수를 늘렸을 때

### 풀이

합성곱 계층의 파라미터 수 = (필터 크기 × 입력 채널 수 × 필터 수) + 필터 수

| 조건 변화 | 파라미터 수 | 이유 |
|---|---|---|
| 특징 탐지·요약·조합 반복 횟수 증가 | **늘어난다** | 합성곱 계층 자체가 늘어난다. 다만 특징 지도가 작아져 분류기의 입력이 줄면 전체는 오히려 줄기도 한다. |
| 합성곱 필터 크기 증가 (3×3 → 5×5) | **늘어난다** | 필터 하나의 가중치가 9개에서 25개로 늘어난다. |
| 풀링 커널 크기 증가 | **변화 없다** | 풀링은 학습 파라미터가 없다. 단 특징 지도가 작아져 **분류기의 선형 계층 파라미터는 줄어든다**. |
| 합성곱 필터 수 증가 | **늘어난다** | 필터 수에 비례해 늘고, 다음 계층의 입력 채널도 늘어 그쪽 파라미터도 함께 커진다. |

핵심은 **풀링은 파라미터가 없다**는 점, 그리고 합성곱 계층의 변화가 **다음 계층의 입력 크기까지 바꿔** 간접적인 영향을 준다는 점이다.

In [ ]:
def n_param(m): return sum(p.numel() for p in m.parameters())
print(f'3x3 필터 32개 (입력 1채널): {n_param(nn.Conv2d(1, 32, 3)):,}개')
print(f'5x5 필터 32개 (입력 1채널): {n_param(nn.Conv2d(1, 32, 5)):,}개')
print(f'3x3 필터 64개 (입력 1채널): {n_param(nn.Conv2d(1, 64, 3)):,}개')
print(f'3x3 필터 32개 (입력 32채널): {n_param(nn.Conv2d(32, 32, 3)):,}개')
print(f'최대 풀링 계층: {n_param(nn.MaxPool2d(2))}개')